# Build widget-only HTMLs

For each city, runs a single-cell notebook that builds the linked deck.gl + Celldega Clustergram pair, then exports to HTML with code cells hidden via `HTMLExporter(exclude_input=True)`.

We **don't** use `embed_minimal_html` here because its requireJS-based loader doesn't bundle anywidget's `_esm` module — the resulting HTML loads correctly in JupyterLab but errors out when opened standalone in a browser. nbconvert embeds anywidget properly, so we lean on that and just hide the code.

In [1]:
import time
from pathlib import Path

import nbformat
from nbconvert import HTMLExporter
from nbconvert.preprocessors import ExecutePreprocessor

ROOT = Path.cwd()
TIMEOUT_S = 1800

CITIES = [
    {'label': 'NYC',     'city': 'nyc',     'year': 2026, 'month': 3,    'n_clusters': 150},
    {'label': 'Boston',  'city': 'boston',  'year': 2026, 'month': 3,    'n_clusters': 100},
    {'label': 'DC',      'city': 'dc',      'year': 2026, 'month': 3,    'n_clusters': 100},
    {'label': 'Chicago', 'city': 'chicago', 'year': 2026, 'month': None, 'n_clusters': 100},
]
print('Will build:', [c['label'] for c in CITIES])

Will build: ['NYC', 'Boston', 'DC', 'Chicago']


## Build each widget-only HTML

In [2]:
CELL_TEMPLATE = '''\
from bike_network_traffic import silence_warnings
silence_warnings()

from ipywidgets import HBox, Layout
from bike_network_traffic import (
    get_bike_data, make_station_clustergram, make_flow_widget, link_flow_to_clustergram,
)

ds = get_bike_data({city!r}, year={year}, month={month!r})
_, cgm, cluster_map = make_station_clustergram(ds.transition_prob, n_clusters={n_clusters})
flow = make_flow_widget(ds.stations, ds.transition_prob, cluster_map, height=700)
link_flow_to_clustergram(flow, cgm)
flow.layout = Layout(width='560px', height='700px')
cgm.layout  = Layout(width='720px', height='700px')
HBox([flow, cgm])
'''

def build_one(label, city, year, month, n_clusters):
    out = ROOT / f'{label}_widget.html'
    print(f'\n=== {label} ===', flush=True)
    t0 = time.time()

    nb = nbformat.v4.new_notebook()
    nb.cells = [nbformat.v4.new_code_cell(
        CELL_TEMPLATE.format(city=city, year=year, month=month, n_clusters=n_clusters)
    )]
    nb.metadata = {
        'kernelspec': {'display_name': 'Python 3 (ipykernel)',
                       'language': 'python', 'name': 'python3'},
        'language_info': {'name': 'python'},
    }

    print('  executing...', flush=True)
    ep = ExecutePreprocessor(timeout=TIMEOUT_S, kernel_name='python3')
    ep.preprocess(nb, {'metadata': {'path': str(ROOT)}})

    exporter = HTMLExporter()
    exporter.exclude_input = True
    exporter.exclude_input_prompt = True
    exporter.exclude_output_prompt = True
    exporter.embed_images = True
    body, _ = exporter.from_notebook_node(nb)
    out.write_text(body, encoding='utf-8')

    size_mb = out.stat().st_size / 1e6
    dt = time.time() - t0
    print(f'  -> {out.name} ({size_mb:.1f} MB) in {dt:.1f}s', flush=True)
    return out

results = {}
for spec in CITIES:
    try:
        results[spec['label']] = ('ok', build_one(**spec))
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}', flush=True)
        results[spec['label']] = ('error', e)


=== NYC ===
  executing...
  -> NYC_widget.html (16.9 MB) in 21.6s

=== Boston ===
  executing...
  -> Boston_widget.html (13.6 MB) in 11.6s

=== DC ===
  executing...
  -> DC_widget.html (14.0 MB) in 12.6s

=== Chicago ===
  executing...
  -> Chicago_widget.html (14.8 MB) in 14.9s


## Summary

In [3]:
for label, (status, info) in results.items():
    if status == 'ok':
        print(f'  OK   {label:<8} -> {info.name}')
    else:
        print(f'  FAIL {label:<8} -> {type(info).__name__}: {info}')

  OK   NYC      -> NYC_widget.html
  OK   Boston   -> Boston_widget.html
  OK   DC       -> DC_widget.html
  OK   Chicago  -> Chicago_widget.html
